## 1. Prepare the Flipkart Price Sequence

In [1]:
import numpy as np

# Sample price of a product on Flipkart for 10 days
# This is made-up data for practice
prices = np.array([
    24999, 25200, 25150, 25500, 25800,
    25700, 26000, 26300, 26150, 26500
], dtype=np.float32)

print("Prices:")
print(prices)
print("Number of days:", len(prices))


Prices:
[24999. 25200. 25150. 25500. 25800. 25700. 26000. 26300. 26150. 26500.]
Number of days: 10


In [2]:
# We will use the previous 3 days to predict the next day's price

window_size = 3

X = []
y = []

for i in range(len(prices) - window_size):
    X.append(prices[i:i + window_size])
    y.append(prices[i + window_size])

X = np.array(X)
y = np.array(y)

print("X before reshaping:")
print(X)
print()
print("y:")
print(y)


X before reshaping:
[[24999. 25200. 25150.]
 [25200. 25150. 25500.]
 [25150. 25500. 25800.]
 [25500. 25800. 25700.]
 [25800. 25700. 26000.]
 [25700. 26000. 26300.]
 [26000. 26300. 26150.]]

y:
[25500. 25800. 25700. 26000. 26300. 26150. 26500.]


In [3]:
# LSTM input format is:
# (number of samples, time steps, number of features)

X = X.reshape((X.shape[0], X.shape[1], 1))

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

print("\nFirst sample:")
print(X[0].flatten())
print("Target:", y[0])


Shape of X: (7, 3, 1)
Shape of y: (7,)

First sample:
[24999. 25200. 25150.]
Target: 25500.0


### Why do we reshape X?

An LSTM expects data in three dimensions:

`(samples, time steps, features)`

Here, each sample contains 3 previous days, and there is only one feature: the product price.

## 2. Build and Train the LSTM Model

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

tf.random.set_seed(42)

# Keep the last sample for a simple test
X_train = X[:-1]
y_train = y[:-1]

X_test = X[-1:]
y_test = y[-1:]

lstm_model = Sequential([
    LSTM(32, input_shape=(window_size, 1)),
    Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

lstm_model.summary()


C:\Users\devds\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 32)                  │           4,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,385 (17.13 KB)

 Trainable params: 4,385 (17.13 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
import time

start_time = time.time()

lstm_history = lstm_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=2,
    verbose=1
)

lstm_time = time.time() - start_time

print(f"\nLSTM training time: {lstm_time:.4f} seconds")


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 671331264.0000
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 671330112.0000
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 671328832.0000
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 671327680.0000
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 671326464.0000
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 671325248.0000
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 671324032.0000
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 671322816.0000
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 671321600.0000
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 671320320.0000

LSTM training time: 5.0136 seconds


## 3. Predict the Next Price

In [6]:
# Predict the price for the test sequence
lstm_prediction = lstm_model.predict(X_test, verbose=0)

predicted_price = lstm_prediction[0][0]
actual_price = y_test[0]

print(f"Predicted price: ₹{predicted_price:.2f}")
print(f"Actual price:    ₹{actual_price:.2f}")
print(f"Difference:      ₹{abs(predicted_price - actual_price):.2f}")


Predicted price: ₹-0.08
Actual price:    ₹26500.00
Difference:      ₹26500.08


The prediction will not be exactly equal to the actual price because the model has been trained on only a very small fake dataset. The main purpose here is to understand the LSTM workflow.

## 4. Replace LSTM with GRU

In [7]:
from tensorflow.keras.layers import GRU

tf.random.set_seed(42)

gru_model = Sequential([
    GRU(32, input_shape=(window_size, 1)),
    Dense(1)
])

gru_model.compile(
    optimizer="adam",
    loss="mse"
)

start_time = time.time()

gru_history = gru_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=2,
    verbose=1
)

gru_time = time.time() - start_time

print(f"\nGRU training time: {gru_time:.4f} seconds")


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 671330496.0000
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 671327872.0000
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 671325184.0000
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 671322560.0000
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 671319872.0000
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 671317312.0000
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 671314624.0000
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 671311936.0000
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 671309312.0000
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 671306688.0000

GRU training time: 4.9572 seconds


In [8]:
# Prediction using GRU
gru_prediction = gru_model.predict(X_test, verbose=0)

gru_predicted_price = gru_prediction[0][0]

print(f"GRU predicted price: ₹{gru_predicted_price:.2f}")
print(f"Actual price:       ₹{actual_price:.2f}")
print(f"Difference:         ₹{abs(gru_predicted_price - actual_price):.2f}")


GRU predicted price: ₹0.20
Actual price:       ₹26500.00
Difference:         ₹26499.80


In [9]:
# Simple comparison

print("----- LSTM vs GRU -----")
print(f"LSTM training time : {lstm_time:.4f} seconds")
print(f"GRU training time  : {gru_time:.4f} seconds")
print()
print(f"LSTM prediction    : ₹{predicted_price:.2f}")
print(f"GRU prediction     : ₹{gru_predicted_price:.2f}")
print(f"Actual price       : ₹{actual_price:.2f}")


----- LSTM vs GRU -----
LSTM training time : 5.0136 seconds
GRU training time  : 4.9572 seconds

LSTM prediction    : ₹-0.08
GRU prediction     : ₹0.20
Actual price       : ₹26500.00


### LSTM vs GRU

Both models can work with sequence data. GRU has a simpler structure than LSTM, so it can sometimes train faster. However, the result depends on the dataset, hardware, model size, and number of training epochs.

Since this example has only a few fake prices, the training-time difference and predictions should **not** be treated as a serious real-world comparison.

## 5. Reusable Function for Preparing Sequence Data

I asked ChatGPT for a general function that can turn a normal sequence into data that an LSTM can understand. I can also use the same function for things like daily Spotify song play counts, website visits, temperature readings, etc.

In [10]:
def make_lstm_data(sequence, window_size):
    X = []
    y = []

    for i in range(len(sequence) - window_size):
        X.append(sequence[i:i + window_size])
        y.append(sequence[i + window_size])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    # Add the feature dimension for the LSTM
    X = X.reshape(X.shape[0], X.shape[1], 1)

    return X, y


# Example: daily Spotify song play counts
spotify_plays = [120, 135, 128, 150, 162, 170, 165, 180]

spotify_X, spotify_y = make_lstm_data(spotify_plays, 3)

print("X shape:", spotify_X.shape)
print("y shape:", spotify_y.shape)

print("\nFirst input:")
print(spotify_X[0].flatten())

print("First target:", spotify_y[0])


X shape: (5, 3, 1)
y shape: (5,)

First input:
[120. 135. 128.]
First target: 150.0


### How the function works

The function takes the original list and a `window_size`. If the window size is 3, it takes 3 values at a time and uses the next value as the answer.

For example, with Spotify plays `[120, 135, 128, 150]`, the first input is `[120, 135, 128]` and the target is `150`.

It repeats this for the whole list and finally reshapes `X` into three dimensions because that is the format expected by an LSTM.

## Final Conclusion

In this session, I prepared a small sequence of product prices and used it with an LSTM model. I also changed the LSTM to a GRU and compared the training time and predictions.

The important thing is the sequence format:

**Samples → Time Steps → Features**

The same idea can be used for prices, song plays, sales, temperature, website traffic, and other time-based data.